# LoRA intent extraction
This run trains only on catalog-grounded synthetic examples. Do not represent its loss as a production quality metric; replace the dataset with licensed, reviewed examples before publishing results.

In [ ]:
!pip -q install transformers peft datasets accelerate bitsandbytes trl
from google.colab import files
uploaded = files.upload()  # upload intent_train.jsonl

In [ ]:
import json
from datasets import load_dataset
base_model = 'Qwen/Qwen2.5-0.5B-Instruct'
dataset = load_dataset('json', data_files='intent_train.jsonl')['train']
assert len(dataset) >= 100, 'Use a sufficiently sized, reviewed training set.'
def format_row(row):
    prompt = 'Extract product-search intent as JSON. Conversation:\n' + '\n'.join(row['conversation'])
    return {'text': prompt + '\nJSON:\n' + json.dumps(row['target'])}
dataset = dataset.map(format_row)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token
quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(base_model, quantization_config=quantization, device_map='auto')
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'], lora_dropout=0.05, task_type='CAUSAL_LM')
model = get_peft_model(model, lora)

def tokenize(rows):
    return tokenizer(rows['text'], truncation=True, max_length=512)

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
training_args = TrainingArguments(output_dir='intent-lora', num_train_epochs=3, per_device_train_batch_size=2, gradient_accumulation_steps=8, learning_rate=2e-4, logging_steps=10, save_strategy='epoch', report_to='none', fp16=True)
trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_dataset, data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False))
train_result = trainer.train()
trainer.save_model('intent-lora-adapter')
tokenizer.save_pretrained('intent-lora-adapter')
!zip -r intent-lora-adapter.zip intent-lora-adapter
files.download('intent-lora-adapter.zip')

In [ ]:
# Inference only: reload the saved adapter; this does not retrain.
# Colab may preinstall an old torchao that is incompatible with PEFT.
# This FP16 demo does not use torchao, so remove it before loading the adapter.
!pip uninstall -y -q torchao
import os
from google.colab import files
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

adapter_dir = 'intent-lora-adapter'
if not os.path.isdir(adapter_dir):
    print('Upload intent-lora-adapter.zip, then this cell will extract it.')
    files.upload()
    !unzip -q intent-lora-adapter.zip

base_model = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token
# Use fp16 for inference: it is reliable on a Colab T4 and avoids a runtime-specific bitsandbytes dependency.
base = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=torch.float16, device_map='auto')
model = PeftModel.from_pretrained(base, adapter_dir).eval()

def run_demo(query: str) -> str:
    prompt = 'Extract product-search intent as JSON. Conversation:\n' + query + '\nJSON:\n'
    inputs = tokenizer(prompt, return_tensors='pt').to(next(model.parameters()).device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=160, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# Edit this one line and rerun the cell to demo any product-search query.
query = 'Find me a black scarf under $500.'
print('Query:', query)
print('Fine-tuned intent output:\n', run_demo(query))
# After this cell has run, you can call: print(run_demo('your next query'))